In [0]:
%pip install geopandas osmnx shapely plotly

In [0]:
import pandas as pd
import json
import plotly.express as px
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox

spark_df = spark.table("dbr_dev.artemzharkov10_gold.gold_realtime_road_hazard")
df = spark_df.toPandas()

dt_series = pd.to_datetime(df['weather_time'])
df['time_str'] = dt_series.dt.floor('15min').dt.strftime('%Y-%m-%d %H:%M')
df = df.sort_values('time_str')

d_lon = 0.52
d_lat = 0.315
polygons = []
ids = []

unique_grids = df[['ID', 'longitude', 'latitude']].drop_duplicates()

for _, row in unique_grids.iterrows():
    lon, lat = row['longitude'], row['latitude']
    half_lon, half_lat = d_lon / 2, d_lat / 2
    poly = Polygon([
        (lon - half_lon, lat + half_lat),
        (lon + half_lon, lat + half_lat),
        (lon + half_lon, lat - half_lat),
        (lon - half_lon, lat - half_lat)
    ])
    polygons.append(poly)
    ids.append(row['ID'])

grid_gdf = gpd.GeoDataFrame({'ID': ids, 'geometry': polygons}, crs="EPSG:4326")

poland_gdf = ox.geocode_to_gdf("Poland")
clipped_grid = gpd.clip(grid_gdf, poland_gdf)

clipped_grid['geometry'] = clipped_grid['geometry'].simplify(tolerance=0.02)
geojson_grid = json.loads(clipped_grid.to_json())

temp_min = df['soil_temperature_c'].min()
temp_max = df['soil_temperature_c'].max()
precip_max = df['precipitation_mm'].max()

fig_temp = px.choropleth_mapbox(
    df,
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='soil_temperature_c',        
    color_continuous_scale='RdYlBu_r', 
    color_continuous_midpoint=0,       
    range_color=[-5, 30],  
    mapbox_style="carto-positron",    
    zoom=5.2,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.75,                      
    animation_frame='time_str', 
    title='🌡️ Stream Soil Temperature Anomaly (°C)',
    labels={'soil_temperature_c': 'Temp (°C)'}
)

fig_temp.update_traces(marker_line_width=0.5, marker_line_color='rgba(0,0,0,0.1)') 
fig_temp.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    paper_bgcolor='white', 
    font_color='black'
)

fig_precip = px.choropleth_mapbox(
    df,
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='precipitation_mm',        
    color_continuous_scale='dense',    
    range_color=[0, precip_max + 0.1], 
    mapbox_style="carto-positron", 
    zoom=5.2,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.8,                       
    animation_frame='time_str',    
    title='❄️ Stream Hourly Precipitation (mm)',
    labels={'precipitation_mm': 'Precip (mm)'}
)

fig_precip.update_traces(marker_line_width=0.5, marker_line_color='rgba(0,0,0,0.1)')
fig_precip.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    paper_bgcolor='white',
    font_color='black'
)

path_temp = "/Volumes/dbr_dev/artemzharkov10_gold/html_maps/Stream_temperature_map.html"
path_precip = "/Volumes/dbr_dev/artemzharkov10_gold/html_maps/Stream_precipitation_map.html"

df = df[['ID', 'time_str', 'soil_temperature_c', 'precipitation_mm']]

fig_temp.write_html(path_temp, include_plotlyjs='cdn')
fig_precip.write_html(path_precip, include_plotlyjs='cdn')

print(f"Files saved in Volume:\n1. {path_temp}\n2. {path_precip}")

In [0]:
import pandas as pd
import json
import plotly.express as px
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox

spark_df = spark.table("dbr_dev.artemzharkov10_gold.gold_realtime_road_hazard")
df = spark_df.toPandas()

dt_series = pd.to_datetime(df['weather_time'])
df['time_str'] = dt_series.dt.floor('15min').dt.strftime('%Y-%m-%d %H:%M')
df = df.sort_values('time_str')

d_lon = 0.52
d_lat = 0.315
polygons = []
ids = []

unique_grids = df[['ID', 'longitude', 'latitude']].drop_duplicates()

for _, row in unique_grids.iterrows():
    lon, lat = row['longitude'], row['latitude']
    half_lon, half_lat = d_lon / 2, d_lat / 2
    
    poly = Polygon([
        (lon - half_lon, lat + half_lat),
        (lon + half_lon, lat + half_lat),
        (lon + half_lon, lat - half_lat),
        (lon - half_lon, lat - half_lat)
    ])
    polygons.append(poly)
    ids.append(row['ID'])

grid_gdf = gpd.GeoDataFrame({'ID': ids, 'geometry': polygons}, crs="EPSG:4326")

# Clip grid to Poland's boundaries and OPTIMIZE GEOMETRY
poland_gdf = ox.geocode_to_gdf("Poland")
clipped_grid = gpd.clip(grid_gdf, poland_gdf)
clipped_grid['geometry'] = clipped_grid['geometry'].simplify(tolerance=0.02)
geojson_grid = json.loads(clipped_grid.to_json())

cols_for_map = ['ID', 'time_str', 'hazard_risk', 'weather_claster', 'soil_temperature_c', 'precipitation_mm']
df_plot = df[cols_for_map]

transparent_scale = [
    [0.0, 'rgba(0, 0, 0, 0.0)'],      
    [0.01, 'rgba(65, 105, 225, 0.5)'], 
    [0.5, 'rgba(255, 165, 0, 0.75)'], 
    [1.0, 'rgba(255, 0, 0, 0.9)']      
]


fig_risk = px.choropleth_mapbox(
    df_plot, 
    geojson=geojson_grid,
    locations='ID',                  
    featureidkey='properties.ID',    
    color='hazard_risk',        
    color_continuous_scale=transparent_scale, 
    range_color=[0.94, 1.5],
    mapbox_style="carto-positron",   
    zoom=5.2,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.75,
    animation_frame='time_str', 
    title='⚠️ Stream Road Hazard Risk Map',
    labels={'hazard_risk': 'Hazard Risk'},
    hover_data={'weather_claster': True, 'soil_temperature_c': True, 'precipitation_mm': True} 
)

fig_risk.update_traces(marker_line_width=0.5, marker_line_color='rgba(0,0,0,0.1)')
fig_risk.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    paper_bgcolor='white',
    font_color='black'
)

path_risk = "/Volumes/dbr_dev/artemzharkov10_gold/html_maps/Stream_hazard_risk_map.html"

fig_risk.write_html(path_risk, include_plotlyjs='cdn')

print(f"File saved in Volume:\n1. {path_risk}")
print("To download, go to Catalog -> dbr_dev -> artemzharkov10_gold -> html_maps")